##**What problem are we solving?**




> Add blockquote


When training large models:

- Model does not fit in GPU memory

- Training becomes slow or crashes
- We need to split work across GPUs




Both **DDP (Distributed Data Parallel) and FSDP (Fully Sharded Data Parallel)** are PyTorch techniques used to train models across multiple GPUs / machines.

They help:

- Speed up training

- Fit larger models

- Use GPU memory efficiently


| Concept      | DDP                     | FSDP                         |
| ------------ | ----------------------- | ---------------------------- |
| Model copy   | FULL model on every GPU | Model is **sharded** (split) |
| Memory usage | ❌ High                  | ✅ Low                        |
| Speed        | Faster                  | Slightly slower              |
| Used for     | Small–medium models     | Large / LLM models           |
| Example      | 1B params               | 7B+ params                   |


| Feature            | DDP                 | FSDP            |
| ------------------ | ------------------- | --------------- |
| Model copy per GPU | Full                | Sharded         |
| Memory usage       | High                | Very Low        |
| Speed              | Faster              | Slightly slower |
| Complexity         | Simple              | Complex         |
| Best for           | Small–Medium models | Large / LLMs    |
| Setup difficulty   | Easy                | Medium–Hard     |




**DDP copies the full model on every GPU, then splits the data across GPUs.**
Each GPU:

- Has the entire model

- Processes different batches
:
- Syncs gradients after every step

Simple analogy

- Imagine 4 chefs:

- Each chef has the entire recipe book

- Each cooks different dishes

- At the end, they share improvements

- GPU 0 → Full Model + Batch 1
- GPU 1 → Full Model + Batch 2
- GPU 2 → Full Model + Batch 3
- GPU 3 → Full Model + Batch 4


Gradients are synchronized using All-Reduce.

✅ Pros

✔ Simple
✔ Stable
✔ Fast communication
✔ Best for small–medium models

❌ Cons

❌ Each GPU stores full model
❌ Memory-heavy

In [4]:
#Basic Model (same for both)
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(512, 1024)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(1024, 512)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        return self.layer2(x)


In [5]:
# Dummy Data (Fake Training Data)
def get_data(batch_size=8):
    x = torch.randn(batch_size, 512)
    y = torch.randn(batch_size, 512)
    return x, y



In [7]:
# !pip install torch torchvision accelerate --quiet


In [8]:
import torch
import torch.nn as nn

# What this does:

# torch → main PyTorch library

# torch.nn → used to define neural networks

In [9]:
import torch.distributed as dist
# This enables multi-GPU communication
# Without this → GPUs cannot talk to each other.

In [10]:
from torch.nn.parallel import DistributedDataParallel as DDP
# Why?

# This is PyTorch’s DDP wrapper that:

# - Syncs gradients across GPUs

# - Makes sure all models stay in sync

In [11]:
import os


In [12]:
# STEP 1: Imports
# =====================================================

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.wrap import size_based_auto_wrap_policy
from functools import partial
import os



In [13]:

def setup(rank=0, world_size=1):
    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12355"

    if not dist.is_initialized():
        dist.init_process_group(
            backend="gloo",
            rank=rank,
            world_size=world_size
        )


def cleanup():
    if dist.is_initialized():
        dist.destroy_process_group()

In [14]:
#  STEP 5: DDP TRAINING (SAFE)
# =====================================================

def train_ddp():
    print("\n🚀 Running DDP (Single GPU Safe Mode)")

    setup(rank=0, world_size=1)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = SimpleModel().to(device)
    model = DDP(model)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    for step in range(3):
        x, y = get_data()
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        output = model(x)
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()

        print(f"[DDP] Step {step} | Loss: {loss.item():.4f}")

    cleanup()


##**FSDP — Fully Sharded Data Parallel**

✅ What it does

FSDP splits (shards):

- model parameters

- gradients

- optimiser states across GPUs.

Each GPU holds only a fraction of the model.


🔍 Analogy

Instead of every chef having the full recipe book:

- Each chef has only a few pages

- They exchange pages when needed


GPU 0 → parameters 0–25%
GPU 1 → parameters 25–50%
GPU 2 → parameters 50–75%
GPU 3 → parameters 75–100%


During forward/backward:

- Parameters are gathered just-in-time

- Then released to save memory


✅ Pros

✔ Massive memory savings
✔ Enables training huge models
✔ Works well with mixed precision

❌ Cons

❌ More complex
❌ Slightly slower than DDP
❌ Harder debugging

In [15]:
def train_fsdp():
    print("\n🚀 Running FSDP (Single GPU Safe Mode)")

    setup(rank=0, world_size=1)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = SimpleModel().to(device)

    # ✅ CORRECT way
    auto_wrap_policy = partial(
        size_based_auto_wrap_policy,
        min_num_params=1e6
    )

    model = FSDP(
        model,
        auto_wrap_policy=auto_wrap_policy
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    for step in range(3):
        x, y = get_data()
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        output = model(x)
        loss = loss_fn(output, y)
        loss.backward()
        optimizer.step()

        print(f"[FSDP] Step {step} | Loss: {loss.item():.4f}")

    cleanup()


In [16]:
# STEP 7: RUN
# =====================================================

train_ddp()
train_fsdp()


🚀 Running DDP (Single GPU Safe Mode)
[DDP] Step 0 | Loss: 1.0502
[DDP] Step 1 | Loss: 1.0598
[DDP] Step 2 | Loss: 1.1041

🚀 Running FSDP (Single GPU Safe Mode)
[FSDP] Step 0 | Loss: 1.1289
[FSDP] Step 1 | Loss: 1.0665
[FSDP] Step 2 | Loss: 1.0573


/usr/local/lib/python3.12/dist-packages/torch/distributed/fsdp/_init_utils.py:430: UserWarning: FSDP is switching to use `NO_SHARD` instead of ShardingStrategy.FULL_SHARD since the world size is 1.
  warnings.warn(
